# Member 5: modelling run on Colab

Runs `python -m src.run_models` (baselines, tuning, comparison, **single** test evaluation, SHAP, secondary target, error analysis), then the rolling-origin step, then builds the slides. Put these two files in Google Drive folder **`MyDrive/m5/`** first:

1. `m5_colab_bundle.zip` (code + weekly counts; made by Claude, see the repo message)
2. `questions_clean_v1.parquet`

Use *Runtime > Change runtime type* and pick a **GPU (T4)** runtime with high-RAM if you can: XGBoost then trains on the GPU automatically (Random Forest, TF-IDF and SVD stay on the CPU; TPUs are not used). Run the cells in order. **The test period is evaluated once**: do not re-run step 4 after it has finished; if Colab disconnects *during* step 4, delete `/content/repo/reports/review1/m5` and run step 4 again.

## 1. Check the machine

In [ ]:
import os, subprocess
print('CPU cores:', os.cpu_count())
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
!nvidia-smi -L || echo 'No GPU: XGBoost will run on the CPU (slower but fine)'

## 2. Get the files from Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, zipfile, pathlib
D = pathlib.Path('/content/drive/MyDrive/m5')
R = pathlib.Path('/content/repo')
shutil.rmtree(R, ignore_errors=True)
zipfile.ZipFile(D / 'm5_colab_bundle.zip').extractall(R)
(R / 'data/processed').mkdir(parents=True, exist_ok=True)
shutil.copy(D / 'questions_clean_v1.parquet', R / 'data/processed/questions_clean_v1.parquet')
print(sorted(p.name for p in R.rglob('*') if p.is_file())[:40])

In [ ]:
!pip install -q -U shap python-pptx pyarrow xgboost pyyaml

## 3. Quick check: load the data and build features (about 1 minute)
Expect **302,026** rows loaded and **293,510** feature rows.

In [ ]:
%cd /content/repo
!python -W ignore - <<'EOF'
import pandas as pd
from src import build_features as bf
q = pd.read_parquet('data/processed/questions_clean_v1.parquet')
f = bf.build_features(q, pd.read_csv('data/processed/weekly_counts.csv'))
print(len(q), len(f), f.attrs['dropped_no_history'])
EOF

## 4. Main run (tuning, comparison, the single test evaluation, SHAP, secondary target, error analysis)
Leaves out the slow rolling-origin step. Takes roughly 30 to 90 minutes on Colab CPU.

In [ ]:
import os
os.environ['M5_JOBS'] = str(os.cpu_count())
os.environ['M5_DEVICE'] = 'auto'   # GPU for XGBoost if available
!python -W ignore -m src.run_models --no-rolling 2>&1 | tee /content/run_main.log

Save a copy to Drive right away, so a disconnect cannot lose it.

In [ ]:
import shutil
shutil.make_archive('/content/drive/MyDrive/m5/m5_results_main', 'zip', '/content/repo/reports/review1/m5')
print('saved m5_results_main.zip to Drive')

## 5. Rolling-origin evaluation (slow; 11 half-year windows, each refits the model twice)
Uses the saved settings of step 4. It does not touch the test period.

In [ ]:
!python -W ignore -m src.run_models --rolling-only 2>&1 | tee /content/run_rolling.log

## 6. Slides and findings text

In [ ]:
!python -m src.make_slides

## 7. Copy everything back to Drive

In [ ]:
import shutil
shutil.make_archive('/content/drive/MyDrive/m5/m5_results_all', 'zip', '/content/repo/reports/review1/m5')
for f in ('m5_slides.pptx', 'm5_findings.md'):
    shutil.copy('/content/repo/reports/review1/' + f, '/content/drive/MyDrive/m5/' + f)
print('Done: m5_results_all.zip, m5_slides.pptx, m5_findings.md are in Drive/m5')